# 示例 08 · LangGraph 的思维方式

**来源：** [LangChain 官方文档 — Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph)

---

## 学习目标

完成本 Notebook 后，你将能够：

1. 将真实业务流程（客服邮件系统）映射为 LangGraph `StateGraph`
2. 设计一个存储原始数据的 `TypedDict` 状态，供所有节点共享
3. 实现四种节点类型：LLM 节点、数据节点、行动节点、用户输入节点
4. 应用三种错误处理策略：`RetryPolicy`、Command 循环恢复、`error_handler`
5. 使用 `interrupt()` 在发送回复前暂停执行等待人工审核
6. 编译并端到端运行完整图

---

## 背景介绍

### 核心思路：业务流程 → 图

任何智能体任务都可以分解为一个有向图：

```
START
  └─► 读取邮件
         └─► 分类意图 ──► 搜索文档 ─┐
                     ──► Bug 追踪   ├─► 起草回复 ──► 人工审核 ──► 发送回复 ──► END
                     ──► 人工审核 ──┘          └──────────────────────────────────► END
```

### 四种节点类型

| 类型 | 作用 | 示例 |
|------|------|------|
| **LLM 节点** | 理解、分析、生成文本 | `classify_intent`、`draft_response` |
| **数据节点** | 读写外部系统 | `search_documentation`、`lookup_customer_history` |
| **行动节点** | 执行有副作用的操作 | `send_reply`、`bug_tracking` |
| **用户输入节点** | 暂停等待人工决策 | `human_review` |

### 状态设计黄金法则

> **存储原始数据，不存格式化文本。**
> 正确：`email_content: str`、`classification: EmailClassification`
> 错误：`formatted_prompt: str`（可从其他字段派生）

### 错误处理矩阵

| 错误类型 | 责任方 | 策略 | LangGraph API |
|----------|--------|------|---------------|
| 暂时性（网络、限流） | 系统 | 自动重试 | `RetryPolicy` |
| LLM 可恢复（工具失败、解析） | LLM | 写入错误，循环回去 | `Command(goto=node)` |
| 用户可修复（缺失信息） | 人工 | 暂停等待输入 | `interrupt()` |
| 重试耗尽后可恢复 | 开发者 | 补偿逻辑 | `error_handler` |
| 未知异常 | 开发者 | 直接抛出 | `raise` |

请**从上到下**依次运行每个单元格。

## 环境配置

In [ ]:
import sys
from pathlib import Path
from typing import TypedDict, Literal

# 添加项目根目录，使 common/ 包可被导入
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt, RetryPolicy
from langgraph.errors import NodeError
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

# 初始化 LLM 和 Langfuse 追踪处理器
llm = create_llm()
_handler = create_langfuse_handler()

def make_config(thread_id: str = "s08", trace_name: str | None = None) -> dict:
    cfg = build_langfuse_config(_handler, session_id=thread_id, trace_name=trace_name or thread_id)
    # LangGraph 需要 configurable.thread_id 来启用检查点（interrupt 必须）
    cfg["configurable"] = {"thread_id": thread_id}
    return cfg

print("✓ 环境配置完成")

---

## 第一部分 · 工作流映射与状态设计

编写任何代码之前，先把业务流程映射为节点，再设计状态。

**状态字段判断标准：**
- 被**多个节点**使用 → 放入状态
- 重新获取**代价高**（API 调用、LLM 推理）→ 放入状态
- 否则 → 保留为节点函数内的局部变量

In [ ]:
# ── 用于结构化 LLM 输出的嵌套类型 ────────────────────────────────────────────
class EmailClassification(TypedDict):
    # 邮件意图：问题/Bug/账单/功能请求/复杂
    intent: Literal["question", "bug", "billing", "feature", "complex"]
    urgency: Literal["low", "medium", "high", "critical"]  # 紧急程度
    topic: str    # 主题关键词
    summary: str  # 一句话摘要

# ── 主图状态 ──────────────────────────────────────────────────────────────────
class EmailAgentState(TypedDict):
    # ── 输入字段（图入口时设置） ─────────────────────────
    email_content:    str
    sender_email:     str
    email_id:         str
    # ── LLM 节点写入，后续节点读取 ──────────────────────
    classification:   EmailClassification | None
    # ── 数据节点写入 ─────────────────────────────────────
    search_results:   list[str] | None
    customer_history: dict | None
    # ── 行动节点写入 ─────────────────────────────────────
    draft_response:   str | None
    # ── 消息历史（LLM 上下文） ────────────────────────────
    messages:         list | None

print("✓ 状态类型定义完成")
print("EmailAgentState 字段：", list(EmailAgentState.__annotations__.keys()))

---

## 第二部分 · 节点实现

每个节点都是普通的 Python 函数：接收状态，完成一件事，返回更新。
我们为邮件客服工作流实现全部四种节点类型。

### 步骤 2a — LLM 节点：`read_email` 和 `classify_intent`

In [ ]:
# ── LLM 节点 1：读取邮件 ──────────────────────────────────────────────────────
def read_email(state: EmailAgentState) -> dict:
    # 将原始邮件包装为 HumanMessage，初始化消息历史
    print(f"  [read_email] 处理来自 {state['sender_email']} 的邮件")
    return {
        "messages": [HumanMessage(content=f"处理邮件：{state['email_content'][:80]}...")]
    }

# ── LLM 节点 2：分类意图 ──────────────────────────────────────────────────────
def classify_intent(
    state: EmailAgentState,
) -> Command[Literal["search_documentation", "bug_tracking", "human_review", "draft_response"]]:
    # with_structured_output 强制 LLM 按 EmailClassification 格式返回结果
    structured_llm = llm.with_structured_output(EmailClassification)

    classification_prompt = (
        f"分析以下客服邮件并分类。\n"
        f"邮件：{state['email_content']}\n"
        f"发件人：{state['sender_email']}\n"
        f"请返回：intent（question/bug/billing/feature/complex）、"
        f"urgency（low/medium/high/critical）、topic、summary。"
    )

    classification = structured_llm.invoke(classification_prompt)
    intent  = classification["intent"]
    urgency = classification["urgency"]

    # 路由逻辑：根据意图和紧急程度决定下一个节点
    if intent == "billing" or urgency == "critical":
        goto = "human_review"           # 账单/紧急 → 人工审核
    elif intent in ("question", "feature"):
        goto = "search_documentation"   # 问题/功能 → 搜索文档库
    elif intent == "bug":
        goto = "bug_tracking"           # Bug → 创建追踪工单
    else:
        goto = "draft_response"         # 其他 → 直接起草回复

    print(f"  [classify_intent] intent={intent!r}, urgency={urgency!r} → {goto}")

    # Command 同时完成两件事：更新分类结果 + 指定下一节点
    return Command(update={"classification": classification}, goto=goto)

print("✓ LLM 节点定义完成")

### 步骤 2b — 数据节点：`search_documentation` 和 `bug_tracking`

In [ ]:
# ── 数据节点 1：搜索文档 ─────────────────────────────────────────────────────
def search_documentation(
    state: EmailAgentState,
) -> Command[Literal["draft_response"]]:
    # 数据节点：从知识库检索相关文档（RetryPolicy 保护暂时性网络故障）
    classification = state.get("classification") or {}
    query = f"{classification.get('intent', '')} {classification.get('topic', '')}"

    # 模拟知识库（真实项目替换为 RAG 检索）
    kb = {
        'password': [
            '通过「设置 > 安全 > 修改密码」重置密码',
            '密码需至少 12 位，包含大小写字母、数字和符号',
        ],
        'billing': [
            '发票在「账户 > 账单」页面可查看',
            '退款将在 5-7 个工作日内处理完成',
        ],
        'feature': [
            '功能建议请提交至 feedback.example.com',
            '高票建议将优先纳入产品路线图',
        ],
    }
    results = []
    for key, docs in kb.items():
        if key in query.lower():
            results.extend(docs)
    if not results:
        results = ['暂无针对性文档，将使用通用处理准则。']

    print(f"  [search_documentation] 找到 {len(results)} 条文档，查询：{query[:40]!r}")
    return Command(update={"search_results": results}, goto="draft_response")

# ── 数据节点 2：Bug 追踪 ─────────────────────────────────────────────────────
def bug_tracking(
    state: EmailAgentState,
) -> Command[Literal["draft_response"]]:
    # 行动节点：创建 Bug 工单（带 RetryPolicy 保护）
    ticket_id = f"BUG-{hash(state['email_content']) % 90000 + 10000}"
    print(f"  [bug_tracking] 已创建工单 {ticket_id}")
    return Command(
        update={"search_results": [f"Bug 工单 {ticket_id} 已创建并分配给工程团队。"]},
        goto="draft_response",
    )

print("✓ 数据节点定义完成")

### 步骤 2c — 行动节点：`draft_response` 和 `send_reply`

In [ ]:
# ── 行动节点 1：起草回复 ─────────────────────────────────────────────────────
def draft_response(
    state: EmailAgentState,
) -> Command[Literal["human_review", "send_reply"]]:
    # 综合分类结果和搜索结果，调用 LLM 起草回复
    classification = state.get("classification") or {}
    intent  = classification.get("intent", "unknown")
    urgency = classification.get("urgency", "medium")

    # 组装提示词上下文：搜索结果 + 客户历史
    context_parts = []
    if state.get("search_results"):
        docs = chr(10).join(f"- {d}" for d in state["search_results"])
        context_parts.append(f"相关文档：\n{docs}")
    if state.get("customer_history"):
        tier = state["customer_history"].get("tier", "标准")
        context_parts.append(f"客户等级：{tier}")

    draft_prompt = (
        f"请为以下客服邮件起草专业回复（用中文）。\n"
        f"邮件内容：{state['email_content']}\n"
        f"意图：{intent}，紧急程度：{urgency}\n"
        + (chr(10).join(context_parts) + chr(10) if context_parts else "")
        + "回复需简洁、友好、有针对性。"
    )

    response = llm.invoke(draft_prompt)

    # 高优先级或复杂问题需要人工审核后再发送
    needs_review = (urgency in ("high", "critical") or intent == "complex")
    goto = "human_review" if needs_review else "send_reply"

    print(f"  [draft_response] needs_review={needs_review} → {goto}")
    return Command(update={"draft_response": response.content}, goto=goto)

# ── 行动节点 2：发送回复 ─────────────────────────────────────────────────────
def send_reply(state: EmailAgentState) -> dict:
    # 最终行动：发送邮件（真实项目替换为邮件 SDK 调用）
    draft = state.get("draft_response", "（无草稿）")
    print(f"  [send_reply] ✉ 已发送给 {state['sender_email']}：{draft[:80]}...")
    return {}

print("✓ 行动节点定义完成")

### 步骤 2d — 用户输入节点：`human_review`

In [ ]:
# ── 用户输入节点：人工审核 ────────────────────────────────────────────────────
def human_review(
    state: EmailAgentState,
) -> Command[Literal["send_reply", "__end__"]]:
    # interrupt() 暂停图执行并等待人工决定
    # checkpointer 已保存完整图状态，可以随时恢复——哪怕几天后
    classification = state.get("classification") or {}

    human_decision = interrupt({
        "email_id":       state.get("email_id", ""),
        "original_email": state.get("email_content", ""),
        "draft_response": state.get("draft_response", ""),
        "urgency":        classification.get("urgency"),
        "intent":         classification.get("intent"),
        "action":         "请审核并批准/编辑/拒绝此回复",
    })

    if human_decision.get("approved"):
        # 使用人工编辑后的版本（若未修改则保留原草稿）
        edited = human_decision.get("edited_response") or state.get("draft_response", "")
        print(f"  [human_review] ✓ 已批准")
        return Command(update={"draft_response": edited}, goto="send_reply")
    else:
        print(f"  [human_review] ✗ 已拒绝 — 丢弃草稿")
        return Command(update={}, goto=END)

print("✓ 用户输入节点定义完成")

---

## 第三部分 · 错误处理模式

LangGraph 提供三种互补机制，让图具备生产级别的弹性：

| 机制 | 配置位置 | 处理场景 |
|------|----------|----------|
| `RetryPolicy` | `add_node(..., retry_policy=...)` | 暂时性故障（网络、限流）|
| `Command` 循环回路 | 节点函数内部 | LLM 可恢复的错误 |
| `error_handler` | `add_node(..., error_handler=...)` | 重试耗尽后的补偿逻辑 |

In [ ]:
# ── 模式 1：RetryPolicy 处理暂时性故障 ───────────────────────────────────────
# 搜索文档节点使用指数退避重试，最多 3 次
search_retry_policy = RetryPolicy(
    max_attempts=3,       # 最多重试 3 次
    initial_interval=1.0, # 第 1 次重试等待 1 秒
    backoff_factor=2.0,   # 指数退避：1s → 2s → 4s
    max_interval=10.0,    # 最长等待不超过 10 秒
)
print("RetryPolicy 配置：", search_retry_policy)

# ── 模式 2：Command 循环处理 LLM 可恢复的错误 ─────────────────────────────────
# 工具调用失败时，将错误信息写回状态，让 LLM 重新决策
def execute_tool_safe(
    state: dict,
) -> Command[Literal["agent", "execute_tool"]]:
    # 模拟 LLM 可恢复的工具错误处理模式
    try:
        result = f"工具结果：{state.get('tool_call', '未知')}"
        return Command(update={"tool_result": result}, goto="agent")
    except Exception as e:
        # 错误信息写入状态 → 路由回 agent 节点让 LLM 调整策略
        return Command(update={"tool_result": f"工具错误：{e}"}, goto="agent")

# ── 模式 3：error_handler 执行补偿逻辑 ────────────────────────────────────────
def payment_node(state: dict) -> dict:
    # 模拟可能失败的支付节点
    raise ConnectionError("支付网关不可达")

def payment_error_handler(state: dict, error: NodeError) -> Command[Literal["finalize"]]:
    # 所有重试耗尽后触发：执行退款等补偿操作
    print(f"  [error_handler] 对 {error.node_id} 执行补偿逻辑")
    return Command(
        update={"status": f"已补偿：{str(error)}"},
        goto="finalize",
    )

print("✓ 错误处理模式定义完成")

---

## 第四部分 · 图编译与端到端运行

将所有节点连接起来，挂载 `MemorySaver` 检查点器（`interrupt()` 的必要条件），
然后运行两个场景。

**检查点规则：** `interrupt()` 需要检查点器来持久化完整图状态，
使图可以在中断处恢复——哪怕是几天后。

In [ ]:
# ── 构建并编译完整图 ──────────────────────────────────────────────────────────
workflow = StateGraph(EmailAgentState)

# 注册节点；search_documentation 附加 RetryPolicy 防止暂时性故障
workflow.add_node("read_email",      read_email)
workflow.add_node("classify_intent", classify_intent)
workflow.add_node(
    "search_documentation",
    search_documentation,
    retry_policy=RetryPolicy(max_attempts=3, initial_interval=1.0),
)
workflow.add_node("bug_tracking",   bug_tracking)
workflow.add_node("draft_response", draft_response)
workflow.add_node("human_review",   human_review)
workflow.add_node("send_reply",     send_reply)

# 线性边：入口 → read_email → classify_intent
# classify_intent 通过 Command(goto=...) 动态路由，不需要 add_conditional_edges
workflow.add_edge(START, "read_email")
workflow.add_edge("read_email", "classify_intent")
workflow.add_edge("send_reply", END)

# MemorySaver：内存检查点器（interrupt() 必须有 checkpointer）
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

print("✓ 图编译完成")
print("节点列表：", list(app.graph.nodes))

### 步骤 4a — 场景 A：简单问题（无需人工审核）

In [ ]:
# 场景 A：密码重置问题 → search_documentation → draft_response → send_reply
email_a = {
    "email_content": "你好，我忘记了密码无法登录，请问如何重置密码？",
    "sender_email":  "user@example.com",
    "email_id":      "EMAIL-001",
    "classification": None, "search_results": None,
    "customer_history": None, "draft_response": None, "messages": None,
}

print("=== 场景 A：密码重置问题 ===")
result_a = app.invoke(email_a, config=make_config("s08-scenA-cn"))
print(f"\n最终草稿：{result_a['draft_response'][:200]}...")

### 步骤 4b — 场景 B：紧急账单问题（触发人工审核）

In [ ]:
# 场景 B：紧急账单问题 → classify_intent 路由到 human_review → interrupt()
email_b = {
    "email_content": "我这个月被重复扣费了两次！这非常紧急，请立即处理！",
    "sender_email":  "angry@example.com",
    "email_id":      "EMAIL-002",
    "classification": None, "search_results": None,
    "customer_history": None, "draft_response": None, "messages": None,
}

cfg_b = make_config("s08-scenB-cn")

print("=== 场景 B：紧急账单问题 ===")
result_b = app.invoke(email_b, config=cfg_b)

if result_b.get("__interrupt__"):
    pending = result_b["__interrupt__"][0].value
    print(f"\n⚠  需要人工审核！")
    print(f"   意图：{pending['intent']}")
    print(f"   紧急程度：{pending['urgency']}")
    print(f"   草稿：{pending['draft_response'][:120]}...")
else:
    print("无中断 — 邮件已直接发送。")

In [ ]:
# ── 以人工批准恢复执行 ────────────────────────────────────────────────────────
# interrupt() 已持久化完整图状态
# 向相同 thread_id 发送 Command(resume=...) 即可从中断处继续
final_b = app.invoke(
    Command(resume={
        "approved": True,
        "edited_response": (
            "尊敬的客户，非常抱歉给您造成了困扰。"
            "我们已确认重复扣费情况，现已为您立即发起全额退款。"
            "款项将在 3-5 个工作日内退回原支付账户。"
        ),
    }),
    config=cfg_b,   # 相同 thread_id → 从中断处恢复
)
print(f"\n✓ 人工批准后邮件已发送")
print(f"最终回复：{final_b.get('draft_response', '（无）')}...")

---

## 第五部分 · 关键设计原则

### 节点粒度 — 为什么分成独立节点？

```
✓  每个外部服务一个节点  → 独立的 RetryPolicy，互不影响
✓  每次 LLM 调用一个节点 → 可在分类后、行动前检查中间结果
✓  每个副作用一个节点    → 清晰的检查点边界，失败后少重做
✓  每个人工决策一个节点  → interrupt() 精确暂停在此处
```

**检查点洞察：** LangGraph 在每个节点边界保存状态。
节点越细 = 检查点越多 = 失败后需要重复的工作越少。

### 状态管理规则

| 规则 | 理由 |
|------|------|
| 存储原始数据，不存格式化提示词 | 节点函数动态构建提示词，避免重复 |
| 仅存被 ≥2 个节点使用的数据 | 单节点独用的数据保留为局部变量 |
| 存储获取代价高的数据 | 搜索结果、外部 API 响应 |

### `interrupt()` 黄金法则

> 当 `interrupt()` 与其他状态更新组合使用时，**必须先调用 `interrupt()`**。
> `interrupt()` 之前写入的状态会被持久化；之后写入的尚未保存。

```python
def human_review(state):
    decision = interrupt({"draft": state["draft_response"]})  # ← 先暂停
    return Command(update={...}, goto=...)                     # ← 恢复后执行
```

---

## 总结

| 概念 | API | 说明 |
|------|-----|------|
| 图状态 | `TypedDict` | 所有节点共享的原始数据 |
| LLM 节点带路由 | `Command(update={...}, goto="next")` | 一次调用同时更新状态和指定下一节点 |
| 结构化 LLM 输出 | `llm.with_structured_output(TypedDict)` | 返回符合 Schema 的验证字典 |
| 暂时性错误重试 | `RetryPolicy(max_attempts=N, ...)` | 传入 `add_node` 的参数 |
| 补偿处理器 | `error_handler=fn` in `add_node` | 所有重试耗尽后运行 |
| 人工暂停 | `interrupt({payload})` | 保存完整状态；通过 `Command(resume=...)` 恢复 |
| 检查点器 | `MemorySaver()` | `interrupt()` 和多轮状态的必要条件 |

### 核心要点

1. **先画流程图** — 在写代码之前将业务流程映射为节点
2. **状态 = 共享内存** — 只存原始数据；格式化文本在节点内派生
3. **`Command` 一箭双雕** — 一次返回值同时完成状态更新和路由决策
4. **错误类型对应不同策略** — 暂时性重试、LLM 可恢复循环、用户可修复暂停
5. **检查点是免费的** — 节点越细，弹性越强，性能损耗可忽略不计

In [ ]:
print(f'追踪链路查看地址：{get_langfuse_host()}')